# Semana 13 · LangGraph y control explícito (experimento del estudiante)

Mismas secciones que la demo, con predicciones antes de ejecutar y TODOs. El notebook corre de principio a fin aunque los TODO queden sin llenar; llenarlos es la actividad.

## 0. Preparación

In [ ]:
# En Colab instala lo necesario; en local (.venv) todo está y la línea termina en segundos.
# %pip usa el pip del kernel activo (en Colab equivale a !pip).
%pip install -q langgraph langchain-openai openai pydantic transformers torch sentence-transformers

In [ ]:
# Llave de OpenAI. En Colab: panel de secretos (icono de llave) con el nombre OPENAI_API_KEY
# y acceso activado para este notebook. En Jupyter local: variable de entorno antes de abrir.
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass  # fuera de Colab, o sin secreto: se usa la variable de entorno si existe
print("llave cargada" if os.environ.get("OPENAI_API_KEY") else "sin llave: se usará el modelo local")

In [ ]:
# Backend: llave de OpenAI si existe; si no, servidor local compatible (transformers serve).
# Copia reducida de course/shared/local_server.py para que el notebook sea autocontenido.
import os, shutil, subprocess, sys, time, urllib.request

MODELO_LOCAL, PUERTO = "Qwen/Qwen2.5-1.5B-Instruct", 8765

def _vivo(raiz):
    try:
        with urllib.request.urlopen(f"{raiz}/health", timeout=2) as r: return r.status == 200
    except Exception: return False

def ensure_local_server(modelo=MODELO_LOCAL, puerto=PUERTO, espera_s=900):
    raiz = f"http://localhost:{puerto}"
    if _vivo(raiz): return raiz + "/v1"
    exe = shutil.which("transformers") or os.path.join(os.path.dirname(sys.executable), "transformers")
    subprocess.Popen([exe, "serve", "--port", str(puerto), modelo], stdout=open("/tmp/serve.log", "a"),
                     stderr=subprocess.STDOUT, start_new_session=True)
    t0 = time.time()
    while time.time() - t0 < espera_s:
        if _vivo(raiz): return raiz + "/v1"
        time.sleep(3)
    raise RuntimeError("el servidor local no arrancó; revisa /tmp/serve.log")

def backend_config():
    if os.environ.get("OPENAI_API_KEY") and not os.environ.get("OPENAI_BASE_URL"):
        return {"base_url": None, "api_key": os.environ["OPENAI_API_KEY"], "model": os.environ.get("OPENAI_MODEL", "gpt-5-mini"), "local": False}
    base = os.environ.get("OPENAI_BASE_URL") or ensure_local_server()
    return {"base_url": base, "api_key": "local", "model": MODELO_LOCAL, "local": True}

CFG = backend_config()
print("backend:", "servidor local (Chat Completions)" if CFG["local"] else "OpenAI", "| modelo:", CFG["model"])

In [ ]:
# Tools de negocio de Facturio: shared/facturio_tools.py. En local se importa; en Colab se descarga.
import sys, urllib.request
from pathlib import Path
CANDIDATOS = [Path("../../shared"), Path("../shared"), Path("shared")]
ruta = next((p for p in CANDIDATOS if (p / "facturio_tools.py").exists()), None)
if ruta is None:
    url = "https://raw.githubusercontent.com/JunBarrs24/itm-maestria-llms/main/shared/facturio_tools.py"
    Path("shared").mkdir(exist_ok=True)
    urllib.request.urlretrieve(url, "shared/facturio_tools.py")
    ruta = Path("shared")
sys.path.insert(0, str(ruta.resolve()))
import facturio_tools as ft
ft.reiniciar_estado()
FOLIO = list(ft.CARGOS)[0]
print("tools de negocio:", list(ft.TOOLS), "| folio de ejemplo:", FOLIO)

In [ ]:
# LangGraph y LangChain. Constantes de control del grafo.
import json, operator, re, time
from typing import Annotated, Any, Callable, Literal, Optional, TypedDict
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, RetryPolicy, interrupt

MAX_INTENTOS = 2          # reintentos semánticos: cuántas veces vuelve al agente si la validación falla
MAX_RONDAS_TOOLS = 2      # dentro de un nodo agente: cuántas vueltas modelo -> tool -> modelo
print("langgraph listo")

## 1. Predicciones antes de ejecutar

In [ ]:
MI_PREDICCION_LLAVES_AL_MODELO = None   # TODO: de las 17 llaves del state, cuántas llegan al modelo
MI_PREDICCION_ESCALADOS = None          # TODO: de los seis tickets de la semana 12, cuántos terminarán en escalar
MI_PREDICCION_FALLA_SIN_CHECKPOINT = ""  # TODO: una línea: qué se conserva si el nodo agente falla 5 veces sin checkpointer
MI_PREDICCION_FALLA_CON_CHECKPOINT = ""  # TODO: una línea: y con checkpointer
print("predicciones registradas")

## 2. El grafo (mismo código que la demo)

In [ ]:
# El mismo código de examples/grafo_soporte.py, sección 'modelo'.
def modelo(max_tokens: int = 220) -> ChatOpenAI:
    """Un cliente de chat con la misma interfaz en los dos backends."""
    if CFG["local"]:      # servidor local: Chat Completions, modelo pequeño, temperatura fija
        return ChatOpenAI(model=CFG["model"], base_url=CFG["base_url"], api_key=CFG["api_key"] or "local",
                          max_tokens=max_tokens, temperature=0, timeout=180, max_retries=1)
    return ChatOpenAI(model=CFG["model"], max_tokens=max_tokens, timeout=60)   # con llave: OpenAI


LLM = modelo()

# Contador fuera del grafo: cuenta TODAS las llamadas al modelo, incluidas las de intentos
# que el RetryPolicy descartó. El state solo conserva las del intento que terminó.
CONTADOR = {"llamadas": 0, "tokens_in": 0, "tokens_out": 0}


def reiniciar_contador() -> None:
    CONTADOR.update(llamadas=0, tokens_in=0, tokens_out=0)

In [ ]:
# Sección 'tools': las funciones de shared envueltas con @tool, más la inyección de fallas para la demo.
# Inyección de fallas para la demo: {nombre_tool: fallas pendientes}. Cada ejecución
# con fallas pendientes lanza ConnectionError y descuenta una.
FALLAS: dict[str, int] = {}


def inyectar_falla(nombre: str, veces: int) -> None:
    FALLAS[nombre] = veces


def _ejecutar(nombre: str, fn: Callable, **kw) -> dict:
    if FALLAS.get(nombre, 0) > 0:
        FALLAS[nombre] -= 1
        raise ConnectionError(f"{nombre}: falla transitoria inyectada")
    return fn(**kw)


@tool
def consultar_factura(folio: str) -> str:
    """Consulta un cargo por folio (formato PAY- seguido de 8 dígitos). Devuelve monto, fecha, estado y días desde el cargo."""
    return json.dumps(_ejecutar("consultar_factura", ft.consultar_factura, folio=folio), ensure_ascii=False)


@tool
def consultar_plan(usuario_id: str) -> str:
    """Consulta el plan de un usuario (u001 a u005): límites, uso actual y precio."""
    return json.dumps(_ejecutar("consultar_plan", ft.consultar_plan, usuario_id=usuario_id), ensure_ascii=False)


@tool
def buscar_documentacion(pregunta: str) -> str:
    """Busca en la documentación y políticas de Facturio. Úsala para dudas de políticas, límites, plazos y errores."""
    r = _ejecutar("buscar_documentacion", ft.buscar_documentacion, pregunta=pregunta, k=2)
    return json.dumps(r, ensure_ascii=False)


TOOLS_POR_NOMBRE = {t.name: t for t in (consultar_factura, consultar_plan, buscar_documentacion)}

# Least privilege por especialidad (los PERMISOS de la semana 12): cada agente ve solo las
# tools de su categoría. `reembolsar` y `escalar_a_humano` no son tools del modelo: las ejecuta el grafo.
ESPECIALIDADES: dict[str, dict] = {
    "facturacion": {
        "tools": ["consultar_factura", "consultar_plan"],
        "instrucciones": "Eres el especialista de facturación de Facturio. Usa consultar_factura o consultar_plan "
                         "antes de afirmar algo sobre un cargo o un plan. Responde en español, breve.",
    },
    "acceso": {
        "tools": ["buscar_documentacion"],
        "instrucciones": "Eres el especialista de acceso y seguridad de Facturio. Usa buscar_documentacion antes de "
                         "responder. Responde en español, breve, citando la política encontrada.",
    },
    "tecnico": {
        "tools": ["buscar_documentacion"],
        "instrucciones": "Eres el especialista técnico de Facturio. Usa buscar_documentacion con el código de error "
                         "antes de responder. Responde en español, breve. Si no hay solución documentada, dilo.",
    },
    "reembolso": {
        "tools": ["consultar_factura"],
        "instrucciones": "Eres soporte de reembolsos de Facturio. Consulta el cargo con la tool y resume monto, fecha y "
                         "estado. Di que la solicitud pasará a revisión. No afirmes que el reembolso fue realizado.",
    },
    "general": {
        "tools": ["consultar_factura", "consultar_plan", "buscar_documentacion"],
        "instrucciones": "Eres el asistente de la mesa de soporte de Facturio. Usa las tools antes de afirmar algo: "
                         "facturas y planes con las de consulta, políticas y errores con la documentación. "
                         "Responde en español, breve.",
    },
}
Categoria = Literal["facturacion", "acceso", "tecnico", "reembolso", "general"]

In [ ]:
# Sección 'state'.
class EstadoSoporte(TypedDict, total=False):
    # entrada
    mensaje: str
    # decidido por código (clasificar)
    categoria: str
    folio: Optional[str]
    usuario_id: Optional[str]
    monto: Optional[float]
    # producido por el modelo
    respuesta: str
    tool_calls_intento: list[str]                  # las del último intento (se reemplaza)
    tool_calls: Annotated[list[str], operator.add]  # todas las de la corrida (se acumulan)
    ultimo_contexto: list[dict]                    # lo que se envió al modelo, para verlo lado a lado
    # control del grafo: nunca llega al modelo
    intentos: int
    valida: bool
    motivo_invalida: str
    aprobacion: Optional[dict]
    reembolso: Optional[dict]
    traza: Annotated[list[str], operator.add]
    tokens_in: Annotated[int, operator.add]
    tokens_out: Annotated[int, operator.add]

In [ ]:
# Sección 'nodos deterministas'.
RE_FOLIO = re.compile(r"PAY-\d{8}")
RE_USUARIO = re.compile(r"\bu\d{3}\b")
RE_MONTO = re.compile(r"(\d+(?:[.,]\d+)?)\s*(?:pesos|mxn|\$)", re.I)
# Las palabras clave del routing determinista de la semana 12 (topologias.py), con el reembolso separado.
PALABRAS = {
    "facturacion": ("factura", "cobr", "plan ", "plan.", "precio", "pago", "cargo", "descuento"),
    "acceso": ("contraseña", "sesión", "sesion", "2fa", "bloque", "invitaci", "acceso", "entrar"),
    "tecnico": ("error", "e-", "export", "pdf", "api", "webhook", "integraci", "csv"),
}


def clasificar(state: EstadoSoporte) -> dict:
    """Clasificador determinista: el triage de la semana 12 sin modelo. Cero tokens, cero variabilidad."""
    m = state["mensaje"]
    bajo = m.lower()
    folio = RE_FOLIO.search(m)
    usuario = RE_USUARIO.search(m)
    monto = RE_MONTO.search(m)
    if "reembols" in bajo:                       # side effect: camino propio con aprobación humana
        cat = "reembolso"
    elif folio:
        cat = "facturacion"
    elif re.search(r"E-\d{4}", m):
        cat = "tecnico"
    else:
        puntos = {n: sum(1 for k in ks if k in bajo) for n, ks in PALABRAS.items()}
        mejor = max(puntos, key=puntos.get)
        cat = mejor if puntos[mejor] > 0 else "general"   # cuando el código no sabe, el agente general
    return {"categoria": cat, "folio": folio.group(0) if folio else None,
            "usuario_id": usuario.group(0) if usuario else None,
            "monto": float(monto.group(1).replace(",", ".")) if monto else None,
            "intentos": 0, "valida": False, "motivo_invalida": "", "aprobacion": None, "reembolso": None,
            "tool_calls_intento": [], "traza": ["clasificar"]}


RE_CORREO = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
RE_PROMESA = re.compile(r"reembolso (fue|ha sido|se ha|está|esta) (realizado|procesado|aplicado|completado)|"
                        r"hemos reembolsado|se reembols[óo]|ya (fue|está) reembolsad", re.I)


def validar(state: EstadoSoporte) -> dict:
    """Validación determinista de la respuesta del agente. Es el guardrail de salida de la semana 11 como nodo."""
    r = (state.get("respuesta") or "").strip()
    motivo = ""
    if not r:
        motivo = "respuesta vacía"
    elif RE_CORREO.search(r):
        motivo = "la respuesta contiene un correo electrónico"
    elif not state.get("tool_calls_intento"):
        motivo = "respondió sin consultar datos (ninguna tool en este intento)"
    elif RE_PROMESA.search(r) and not state.get("reembolso"):
        motivo = "afirma un reembolso que no se ejecutó"
    return {"valida": motivo == "", "motivo_invalida": motivo, "traza": ["validar"]}


def aprobacion_humana(state: EstadoSoporte) -> dict:
    """Se detiene aquí. El grafo guarda el checkpoint y devuelve el control a la aplicación."""
    decision = interrupt({
        "pregunta": "¿Aprobar el reembolso?",
        "folio": state.get("folio"), "monto": state.get("monto"),
        "resumen_del_agente": (state.get("respuesta") or "")[:300],
    })
    if isinstance(decision, dict):
        return {"aprobacion": decision, "traza": ["aprobacion_humana"]}
    return {"aprobacion": {"aprobado": bool(decision), "quien": "operador"}, "traza": ["aprobacion_humana"]}


def ejecutar_reembolso(state: EstadoSoporte) -> dict:
    """Side effect ejecutado por el grafo, nunca por el modelo. La regla de negocio sigue en shared."""
    ap = state.get("aprobacion") or {}
    if not ap.get("aprobado"):
        return {"reembolso": {"ejecutado": False, "razon": "rechazado por la persona que aprueba"},
                "respuesta": state["respuesta"] + "\n\n[Sistema] La solicitud de reembolso fue rechazada en revisión.",
                "traza": ["ejecutar_reembolso"]}
    folio, monto = state.get("folio"), state.get("monto")
    if not folio or not monto:
        return {"reembolso": {"ejecutado": False, "razon": "faltan folio o monto en la solicitud"},
                "respuesta": state["respuesta"] + "\n\n[Sistema] Falta el folio o el monto; escribe a soporte con ambos.",
                "traza": ["ejecutar_reembolso"]}
    limite_previo = ft.MONTO_MAXIMO_SIN_HUMANO
    ft.MONTO_MAXIMO_SIN_HUMANO = max(limite_previo, monto)   # la aprobación humana sustituye el límite automático
    try:
        res = ft.reembolsar(folio, monto, ap.get("motivo", "solicitud del cliente"))
    finally:
        ft.MONTO_MAXIMO_SIN_HUMANO = limite_previo
    nota = (f"[Sistema] Reembolso aplicado: {res['referencia']} por {res['monto']:.2f}, abono en {res['abono_en']}."
            if res.get("ejecutado") else f"[Sistema] Reembolso no aplicado: {res.get('razon')}.")
    return {"reembolso": res, "respuesta": state["respuesta"] + "\n\n" + nota, "traza": ["ejecutar_reembolso"]}


def escalar(state: EstadoSoporte) -> dict:
    t = ft.escalar_a_humano(motivo=f"validación fallida {state.get('intentos')} veces: {state.get('motivo_invalida')}",
                            folio=state.get("folio"))
    return {"respuesta": f"Tu caso pasó a una persona del equipo (ticket {t['ticket']}). Te contactamos pronto.",
            "traza": ["escalar"]}

In [ ]:
# Sección 'nodos con modelo'.
def contexto_para_modelo(state: EstadoSoporte) -> list:
    """Lo único del state que llega al modelo. Compárese con las llaves del state completo."""
    esp = ESPECIALIDADES[state["categoria"]]
    instr = esp["instrucciones"]
    if state.get("intentos", 0) > 0 and state.get("motivo_invalida"):
        instr += f"\nTu respuesta anterior fue rechazada: {state['motivo_invalida']}. Corrige."
    return [SystemMessage(content=instr), HumanMessage(content=state["mensaje"])]


def _usage(msg: AIMessage) -> tuple[int, int]:
    u = msg.usage_metadata or {}
    return int(u.get("input_tokens", 0)), int(u.get("output_tokens", 0))


def hacer_agente(categoria: str) -> Callable[[EstadoSoporte], dict]:
    """Fábrica: un nodo agente por especialidad, con sus tools y sus instrucciones."""
    esp = ESPECIALIDADES[categoria]
    tools = [TOOLS_POR_NOMBRE[n] for n in esp["tools"]]
    llm_tools = LLM.bind_tools(tools)

    def agente(state: EstadoSoporte) -> dict:
        mensajes = contexto_para_modelo(state)
        t_in = t_out = 0
        calls: list[str] = []
        respuesta = ""
        for ronda in range(MAX_RONDAS_TOOLS + 1):
            ai = (llm_tools if ronda < MAX_RONDAS_TOOLS else LLM).invoke(mensajes)
            a, b = _usage(ai)
            t_in, t_out = t_in + a, t_out + b
            CONTADOR["llamadas"] += 1
            CONTADOR["tokens_in"] += a
            CONTADOR["tokens_out"] += b
            mensajes.append(ai)
            if not ai.tool_calls:
                respuesta = ai.content if isinstance(ai.content, str) else str(ai.content)
                break
            for tc in ai.tool_calls:
                calls.append(tc["name"])
                salida = TOOLS_POR_NOMBRE[tc["name"]].invoke(tc["args"])   # ConnectionError sube al RetryPolicy del nodo
                mensajes.append(ToolMessage(content=salida, tool_call_id=tc["id"]))
        return {"respuesta": respuesta, "tool_calls_intento": calls, "tool_calls": calls,
                "intentos": state.get("intentos", 0) + 1, "tokens_in": t_in, "tokens_out": t_out,
                "ultimo_contexto": [{"role": m.type, "content": (m.content if isinstance(m.content, str) else str(m.content))[:400]}
                                    for m in mensajes],
                "traza": [f"agente_{categoria}"]}

    agente.__name__ = f"agente_{categoria}"
    return agente

In [ ]:
# Sección 'edges condicionales'.
def ruta_por_categoria(state: EstadoSoporte) -> str:
    return f"agente_{state['categoria']}"


def ruta_despues_de_validar(state: EstadoSoporte) -> str:
    if state["valida"]:
        return "aprobacion_humana" if state["categoria"] == "reembolso" else END
    if state["intentos"] < MAX_INTENTOS:
        return f"agente_{state['categoria']}"   # ciclo con límite
    return "escalar"

In [ ]:
# Sección 'construcción'.
def construir_grafo(checkpointer=None, retry_policy: RetryPolicy | None = None):
    """Arma y compila el grafo. Con checkpointer se puede interrumpir y reanudar."""
    if retry_policy is None:
        retry_policy = RetryPolicy(max_attempts=3, initial_interval=0.2, backoff_factor=2.0, retry_on=ConnectionError)
    g = StateGraph(EstadoSoporte)
    g.add_node("clasificar", clasificar)
    for cat in ESPECIALIDADES:
        g.add_node(f"agente_{cat}", hacer_agente(cat), retry_policy=retry_policy)
    g.add_node("validar", validar)
    g.add_node("aprobacion_humana", aprobacion_humana)
    g.add_node("ejecutar_reembolso", ejecutar_reembolso)
    g.add_node("escalar", escalar)

    g.add_edge(START, "clasificar")
    g.add_conditional_edges("clasificar", ruta_por_categoria, [f"agente_{c}" for c in ESPECIALIDADES])
    for cat in ESPECIALIDADES:
        g.add_edge(f"agente_{cat}", "validar")
    g.add_conditional_edges("validar", ruta_despues_de_validar,
                            [f"agente_{c}" for c in ESPECIALIDADES] + ["aprobacion_humana", "escalar", END])
    g.add_edge("aprobacion_humana", "ejecutar_reembolso")
    g.add_edge("ejecutar_reembolso", END)
    g.add_edge("escalar", END)
    return g.compile(checkpointer=checkpointer)


def estado_inicial(mensaje: str) -> dict:
    return {"mensaje": mensaje, "tool_calls": [], "traza": [], "tokens_in": 0, "tokens_out": 0}


def correr(app, mensaje: str, thread_id: str = "demo", recursion_limit: int = 25) -> dict:
    """Invoca el grafo y devuelve el state final (o el parcial, si se interrumpió)."""
    config = {"configurable": {"thread_id": thread_id}, "recursion_limit": recursion_limit}
    t0 = time.perf_counter()
    salida = app.invoke(estado_inicial(mensaje), config)
    salida["latencia_s"] = round(time.perf_counter() - t0, 2)
    return salida


def resumen(state: dict) -> dict:
    return {"categoria": state.get("categoria"), "traza": state.get("traza"), "intentos": state.get("intentos"),
            "valida": state.get("valida"), "motivo": state.get("motivo_invalida") or "",
            "tools": state.get("tool_calls"), "tokens": (state.get("tokens_in", 0), state.get("tokens_out", 0)),
            "latencia_s": state.get("latencia_s"), "interrumpido": "__interrupt__" in state}

## 3. Qué decide el código y qué decide el modelo

Llene la tabla como texto: por cada nodo, quién decide y qué evidencia lo demuestra (por ejemplo, "clasificar: código; regex sin llamada al modelo, tokens = 0").

In [ ]:
QUIEN_DECIDE = {
    "clasificar": "TODO",
    "agente_<categoria>": "TODO",
    "validar": "TODO",
    "ruta_despues_de_validar": "TODO",
    "aprobacion_humana": "TODO",
    "ejecutar_reembolso": "TODO",
    "escalar": "TODO",
}
for k, v in QUIEN_DECIDE.items():
    print(f"{k:<26} {v}")

## 4. Corrida y lado a lado

Corra una consulta y compare el state con el contexto que recibió el modelo. Luego corrija su predicción de la sección 1.

In [ ]:
ft.reiniciar_estado(); reiniciar_contador()
app = construir_grafo(checkpointer=InMemorySaver())
s = correr(app, f"¿Cuál es el estado del cargo {FOLIO}?", thread_id="c1")
print(resumen(s))

In [ ]:
# GRAPH STATE frente a MODEL CONTEXT, lado a lado, de la corrida anterior.
def lado_a_lado(state):
    izquierda = [f"{k}: {str(v)[:38]}" for k, v in state.items() if k != "ultimo_contexto"]
    derecha = [f"{m['role']}: {m['content'][:38]}" for m in state.get("ultimo_contexto", [])]
    print(f"{'GRAPH STATE (' + str(len(izquierda)) + ' llaves)':<52}| MODEL CONTEXT ({len(derecha)} mensajes)")
    print("-" * 52 + "+" + "-" * 50)
    for i in range(max(len(izquierda), len(derecha))):
        a = izquierda[i] if i < len(izquierda) else ""
        b = derecha[i] if i < len(derecha) else ""
        print(f"{a:<52}| {b}")

lado_a_lado(s)
print("\nEl modelo nunca vio:", [k for k in s if k not in ("mensaje",) and k != "ultimo_contexto"][:8], "...")

In [ ]:
LLAVES_QUE_VIO_EL_MODELO = None   # TODO: cuente en la columna derecha qué llaves del state aparecen (mensaje, y ninguna más)
print("predicción:", MI_PREDICCION_LLAVES_AL_MODELO, "| observado:", LLAVES_QUE_VIO_EL_MODELO)
print("predicción de tickets escalados:", MI_PREDICCION_ESCALADOS, "(se comprueba en la sección 5)")

## 5. Corregir el clasificador

Los seis tickets de la semana 12 se clasifican bien. El ticket trampa de abajo no: "reembols" lo manda al camino del reembolso (con aprobación humana y side effect) cuando es una pregunta de política sin folio ni monto. Corra el grafo con la trampa y vea dónde termina. Luego agregue una regla en `clasificar_v2` que la mande a `general` sin romper los seis tickets.

In [ ]:
# El mini gold set de la semana 12: ticket, especialista esperado, tool esperada.
GOLD = [("g1", f"¿Cuál es el estado de la factura {FOLIO}?", "facturacion", "consultar_factura"),
        ("g2", "Consulta el plan del usuario u002 y dime cuántas facturas al mes incluye.", "facturacion", "consultar_plan"),
        ("g3", "El cargo PAY-00012348 me aparece duplicado, ¿me lo van a devolver?", "facturacion", "consultar_factura"),
        ("g4", "No puedo iniciar sesión: el código 2FA nunca llega.", "acceso", "buscar_documentacion"),
        ("g5", "Al exportar a PDF me sale el error E-4001.", "tecnico", "buscar_documentacion"),
        ("g6", "Invité a un colega hace una semana y dice que el enlace de invitación ya no sirve.", "acceso", "buscar_documentacion")]
TRAMPA = "¿Cuántos días tengo para pedir un reembolso según la política?"
for tid, tarea, esperado, esperada in GOLD:
    print(tid, clasificar({"mensaje": tarea})["categoria"], "| esperado:", esperado)
print("trampa ->", clasificar({"mensaje": TRAMPA})["categoria"])
ft.reiniciar_estado()
s = correr(app, TRAMPA, thread_id="trampa")
if "__interrupt__" in s:
    print("se detuvo a pedir aprobación de un reembolso que nadie pidió:", s["__interrupt__"][0].value)
else:
    print(resumen(s))

In [ ]:
def clasificar_v2(state):
    salida = clasificar(state)
    m = state["mensaje"].lower()
    # TODO: una regla que mande a "general" las preguntas de política de reembolso sin folio ni monto. Ejemplo de forma:
    # if salida["categoria"] == "reembolso" and salida["folio"] is None and salida["monto"] is None:
    #     salida["categoria"] = "general"
    return salida

for tid, tarea, esperado, esperada in GOLD:
    print(tid, clasificar_v2({"mensaje": tarea})["categoria"], "| esperado:", esperado)
print("trampa ->", clasificar_v2({"mensaje": TRAMPA})["categoria"])

In [ ]:
# Grafo con el clasificador corregido. Con el TODO vacío, se comporta como el original.
def construir_grafo_v2(checkpointer=None):
    g = StateGraph(EstadoSoporte)
    g.add_node("clasificar", clasificar_v2)
    for cat in ESPECIALIDADES:
        g.add_node(f"agente_{cat}", hacer_agente(cat), retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.2, retry_on=ConnectionError))
    g.add_node("validar", validar); g.add_node("aprobacion_humana", aprobacion_humana)
    g.add_node("ejecutar_reembolso", ejecutar_reembolso); g.add_node("escalar", escalar)
    g.add_edge(START, "clasificar")
    g.add_conditional_edges("clasificar", ruta_por_categoria, [f"agente_{c}" for c in ESPECIALIDADES])
    for cat in ESPECIALIDADES:
        g.add_edge(f"agente_{cat}", "validar")
    g.add_conditional_edges("validar", ruta_despues_de_validar, [f"agente_{c}" for c in ESPECIALIDADES] + ["aprobacion_humana", "escalar", END])
    g.add_edge("aprobacion_humana", "ejecutar_reembolso"); g.add_edge("ejecutar_reembolso", END); g.add_edge("escalar", END)
    return g.compile(checkpointer=checkpointer)

app2 = construir_grafo_v2(checkpointer=InMemorySaver())
ft.reiniciar_estado(); reiniciar_contador()
s = correr(app2, TRAMPA, thread_id="trampa-v2")
print(resumen(s)); print(s["respuesta"][:250])

## 6. Aprobación humana con política

Hoy toda solicitud de reembolso se detiene. Escriba una política: hasta qué monto aprueba el código solo y desde cuál pide a una persona. Implemente `aprobacion_con_politica` y úsela en lugar de `aprobacion_humana`.

In [ ]:
LIMITE_AUTOMATICO = 0.0     # TODO: monto hasta el cual el código aprueba sin interrumpir (0.0 = siempre pregunta)
POLITICA_APROBACION = ""    # TODO: una línea: quién aprueba, qué se registra, qué pasa fuera de horario

def aprobacion_con_politica(state):
    monto = state.get("monto") or 0.0
    if monto <= LIMITE_AUTOMATICO:
        return {"aprobacion": {"aprobado": True, "quien": "politica automatica"}, "traza": ["aprobacion_automatica"]}
    return aprobacion_humana(state)

g = StateGraph(EstadoSoporte)
g.add_node("clasificar", clasificar)
for cat in ESPECIALIDADES:
    g.add_node(f"agente_{cat}", hacer_agente(cat))
g.add_node("validar", validar); g.add_node("aprobacion_humana", aprobacion_con_politica)
g.add_node("ejecutar_reembolso", ejecutar_reembolso); g.add_node("escalar", escalar)
g.add_edge(START, "clasificar")
g.add_conditional_edges("clasificar", ruta_por_categoria, [f"agente_{c}" for c in ESPECIALIDADES])
for cat in ESPECIALIDADES:
    g.add_edge(f"agente_{cat}", "validar")
g.add_conditional_edges("validar", ruta_despues_de_validar, [f"agente_{c}" for c in ESPECIALIDADES] + ["aprobacion_humana", "escalar", END])
g.add_edge("aprobacion_humana", "ejecutar_reembolso"); g.add_edge("ejecutar_reembolso", END); g.add_edge("escalar", END)
app3 = g.compile(checkpointer=InMemorySaver())

ft.reiniciar_estado()
config = {"configurable": {"thread_id": "pol-1"}}
s = app3.invoke(estado_inicial(f"Solicito el reembolso de 300 pesos del cargo {FOLIO}."), config)
if "__interrupt__" in s:
    print("se detuvo (monto > límite):", s["__interrupt__"][0].value["monto"])
    s = app3.invoke(Command(resume={"aprobado": True, "quien": "supervisora"}), config)
else:
    print("aprobado por política automática")
print(s["traza"], "| reembolsos:", dict(ft.REEMBOLSOS))

## 7. Falla con y sin checkpoint

Primero escriba las predicciones de la sección 1. Luego ejecute y compare.

In [ ]:
print("predicción sin checkpoint:", MI_PREDICCION_FALLA_SIN_CHECKPOINT or "(vacía)")
print("predicción con checkpoint:", MI_PREDICCION_FALLA_CON_CHECKPOINT or "(vacía)")
sin_cp = construir_grafo()
inyectar_falla("consultar_factura", 5)
try:
    sin_cp.invoke(estado_inicial(f"¿Cuánto cobró el cargo {FOLIO}?"), {"configurable": {"thread_id": "x"}})
except ConnectionError as e:
    print("sin checkpointer ->", e)
FALLAS.clear()

config = {"configurable": {"thread_id": "f-cp"}}
inyectar_falla("consultar_factura", 5)
try:
    app.invoke(estado_inicial(f"¿Cuánto cobró el cargo {FOLIO}?"), config)
except ConnectionError as e:
    print("con checkpointer ->", e)
print("state guardado, siguiente nodo:", app.get_state(config).next)
FALLAS.clear()
print("reanudado:", resumen(app.invoke(None, config)))

In [ ]:
# TODO: responda en texto. ¿Qué cambiaría si la falla ocurriera en `ejecutar_reembolso` después de mover el dinero?
# ¿Qué pieza de la semana 8 (idempotencia) protege ese caso?
RESPUESTA_FALLA_EN_SIDE_EFFECT = ""
print(RESPUESTA_FALLA_EN_SIDE_EFFECT or "(pendiente)")

## 8. Mejorar el validador

En la demo, el agente de reembolsos escribió "la solicitud ha sido procesada" antes de la aprobación y `validar` no lo detectó. Agregue un patrón a `RE_PROMESA_V2` que lo atrape sin rechazar "la solicitud pasará a revisión".

In [ ]:
RE_PROMESA_V2 = re.compile(RE_PROMESA.pattern + r"|TODO_PATRON_NUEVO", re.I)   # TODO: sustituya TODO_PATRON_NUEVO
CASOS = [("La solicitud para reembolsar 300 pesos ha sido procesada.", True),
         ("Tu solicitud pasará a revisión; te avisamos en 2 días.", False),
         ("El reembolso fue realizado con éxito.", True)]
for texto, esperado in CASOS:
    print(f"{bool(RE_PROMESA_V2.search(texto))!s:<6} esperado={esperado!s:<6} {texto}")

## 9. Cierre

Responda en texto: para la mesa de soporte de Facturio, ¿qué modelo de control encaja y por qué? Use la tabla medida de la demo y un dato de este notebook.

In [ ]:
DECISION = ""   # TODO: máximo 120 palabras
print(DECISION or "(pendiente)")